# 03 - Comparaison avec un modèle pré-entraîné (zero-shot CamemBERT)

Le notebook `02` entraîne un modèle **supervisé** (TF-IDF + RandomForest) sur nos données labellisées.
Ici, on compare à un modèle **pré-entraîné** utilisé en **zero-shot** (aucun entraînement sur nos données) :
on donne au modèle les paroles et deux labels candidats ("musique rap" / "chanson de variété française"),
et on regarde lequel il juge le plus probable, via du NLI (Natural Language Inference).

**Objectif** : voir ce qu'on obtient *sans aucune donnée d'entraînement labellisée*, à titre de comparaison.
On s'attend à ce que ce soit moins bon que le modèle supervisé (normal, il n'a jamais vu nos données),
mais ça montre qu'une alternative sans entraînement existe.

Dépendances supplémentaires :
```bash
pip install -e ".[zeroshot]"
```

> Résultat : F1 macro zero-shot = 0.5705 vs 0.965 pour le modèle supervisé (détails en section 5).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, f1_score
from transformers import pipeline

from lyrics_classification.data import load_dataset

RANDOM_STATE = 11  # même seed que le notebook 02, pour comparer sur le même split

c:\Users\latej\Desktop\Projets_github\nltk-lyrics-text-classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Reconstruire le même jeu de test que le notebook 02

On refait le split groupé par artiste (`StratifiedGroupKFold`, même `random_state`) pour obtenir
exactement le même jeu de test (6 artistes jamais vus à l'entraînement du modèle supervisé),
afin que la comparaison soit sur les mêmes exemples.

In [2]:
df = load_dataset()
X = df["lyrics"]
groups = df["auteur"]
y = df["genre"].astype(str).map({"VF": 0, "Rap": 1}).astype(int)

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
_, test_idx = next(splitter.split(X, y, groups=groups))

X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]
print("Test:", X_test.shape)
print("Artistes test :", sorted(groups.iloc[test_idx].unique()))

Test: (500,)
Artistes test : ['Diam_s', 'Johnny Hallyday', 'Kaaris', 'Nekfeu', 'Sinik', 'Vald']


## 2) Échantillonner le test set

L'inférence zero-shot avec un modèle transformer est bien plus lente que TF-IDF + RandomForest.
On échantillonne un sous-ensemble (stratifié sur les deux classes) plutôt que les 500 chansons du test
complet, pour garder un temps d'exécution raisonnable. Augmente `N_SAMPLES` si besoin d'une estimation
plus précise.

In [3]:
sample_df = pd.DataFrame({"lyrics": X_test.values, "y_true": y_test.values})
print(sample_df["y_true"].value_counts())

y_true
0    316
1    184
Name: count, dtype: int64


## 3) Classification zero-shot

Modèle : [`cmarkea/distilcamembert-base-nli`](https://huggingface.co/cmarkea/distilcamembert-base-nli),
une version distillée de CamemBERT fine-tunée pour le NLI en français (assez légère pour tourner sur CPU).

Les paroles sont passées en entier au modèle : c'est le tokenizer qui tronque en interne à sa limite de
séquence (~512 tokens). Pour les chansons longues, le modèle ne "voit" donc que leur début, pas le texte
complet (limite à garder en tête pour l'interprétation, cf. section 5).

In [4]:
import sentencepiece

classifier = pipeline("zero-shot-classification", model="cmarkea/distilcamembert-base-nli")

# Labels candidats en français, tels que soumis au modèle NLI (il n'a jamais vu ces labels
# à l'entraînement, on lui demande juste "quelle hypothèse est la plus probable ?").
candidate_labels = ["musique rap", "chanson de variété française"]
label_to_int = {"musique rap": 1, "chanson de variété française": 0}

predictions = []
for lyrics in sample_df["lyrics"]:
    # Paroles passées en entier (pas de troncature manuelle) : le tokenizer tronque lui-même
    # en interne à la limite de tokens du modèle (~512 tokens pour ce CamemBERT distillé).
    # Pour les chansons longues (Rap notamment), le modèle ne "voit" donc que le début du
    # texte -> limite connue de cette comparaison, pas un bug.
    result = classifier(str(lyrics), candidate_labels=candidate_labels)
    top_label = result["labels"][0]  # label jugé le plus probable par le modèle
    predictions.append(label_to_int[top_label])

sample_df["y_pred_zeroshot"] = predictions

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2977.44it/s]


## 4) Évaluation et comparaison au modèle supervisé

In [5]:
# F1 macro : même métrique principale que le notebook 02, pour une comparaison directe.
f1_macro_zeroshot = f1_score(sample_df["y_true"], sample_df["y_pred_zeroshot"], average="macro")
print("F1 macro (zero-shot, échantillon de", len(sample_df), "exemples) :", round(f1_macro_zeroshot, 4))
print()
print(classification_report(sample_df["y_true"], sample_df["y_pred_zeroshot"], target_names=["VF", "Rap"]))

F1 macro (zero-shot, échantillon de 500 exemples) : 0.5705

              precision    recall  f1-score   support

          VF       0.72      0.53      0.61       316
         Rap       0.45      0.66      0.53       184

    accuracy                           0.57       500
   macro avg       0.59      0.59      0.57       500
weighted avg       0.62      0.57      0.58       500



## 5) Conclusion

Résultats sur les **500 chansons du test set** (les mêmes que le notebook 02, 316 VF / 184 Rap) :

| Modèle | Entraîné sur nos données ? | F1 macro (test) |
|---|---|---|
| TF-IDF + RandomForest (supervisé) | Oui | **0.965** |
| CamemBERT distillé (zero-shot, NLI) | Non | **0.5705** |

Détail zero-shot :
```
              precision    recall  f1-score   support

          VF       0.72      0.53      0.61       316
         Rap       0.45      0.66      0.53       184

    accuracy                           0.57       500
   macro avg       0.59      0.59      0.57       500
weighted avg       0.62      0.57      0.58       500
```

**Interprétation :**
- Le zero-shot capte un vrai signal : il identifie correctement 66% des chansons Rap (recall) sans avoir
  jamais vu nos données, ce qui n'a rien d'évident pour un modèle généraliste utilisé hors de son cadre
  d'entraînement (NLI, pas classification de genre musical).
- Il reste cependant très loin du modèle supervisé (0.57 vs 0.965) - écart logique et attendu : il n'a
  jamais vu notre distribution de vocabulaire, nos artistes, ni le déséquilibre des classes.
- Point notable : l'accuracy du zero-shot (0.57) est même **inférieure** à la baseline naïve "toujours
  prédire VF" (63,2%, la classe majoritaire). Mais son F1 macro (0.57) lui est **supérieur** à cette même
  baseline naïve (qui obtiendrait un F1 macro autour de 0.39, plombée par un F1 nul sur Rap). C'est une
  bonne illustration de pourquoi le F1 macro est la métrique à privilégier ici plutôt que l'accuracy brute.
- Limite méthodologique à garder en tête : les paroles sont passées en entier au modèle, qui tronque en
  interne à ~512 tokens (limite du CamemBERT distillé utilisé) - pour les chansons longues, le modèle ne
  "voit" donc que leur début, pas la chanson complète.

**Conclusion générale** : l'entraînement supervisé apporte une valeur ajoutée considérable sur cette tâche
précise. Le zero-shot reste néanmoins une alternative crédible et rapide à mettre en place quand aucune
donnée labellisée n'est disponible (ex. pour explorer un nouveau genre musical sans exemples annotés).